The goal of this notebook is to study the coauthorship network of Pharmacology

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import aquarel
sns.set_palette("viridis")

plt.rcParams.update({
    "font.family": "serif",
    "text.usetex": True,  # Latex
    "font.serif": ["Computer Modern Roman"],  # ou "Times New Roman"
})

In [2]:
import graphistry
graphistry.register(api=3, protocol='https', server='hub.graphistry.com', personal_key_id="L97MZMMRMW", personal_key_secret="X6U4U2UTL76RGMN2")

In [3]:
import polars as pl

In [4]:
import networkx as nx

In [5]:
works = pl.read_parquet("data_pharmacology/topics_pharma_1.0/works_post_topics.parquet")
works = works.filter(pl.col("year")!=2025)

# Complete network (correction of an error)

it may be interesting to keep the direction of the edges: the direction indicate the importance of the author in the article, the arrow goes from the better positioned author to the worse positioned one in the couple. 

In [6]:
import polars as pl
import itertools

# On suppose que ton DataFrame "works" est déjà un pl.DataFrame

# Liste des colonnes auteurs/pays
author_cols = [f"author_{n}" for n in range(1, 100)]
country_cols = [f"country_{n}" for n in range(1, 100)]

# Transformer en format "long" (unpivot/melt)
long_df = works.melt(
    id_vars=["year"], 
    value_vars=author_cols + country_cols,
    variable_name="var", 
    value_name="value"
)

# Séparer auteurs et pays
authors_long = (
    long_df.filter(pl.col("var").str.starts_with("author_"))
    .with_columns([
        pl.col("var").str.strip_chars("author_").cast(pl.Int64).alias("pos")
    ])
    .rename({"value": "author"})
)

countries_long = (
    long_df.filter(pl.col("var").str.starts_with("country_"))
    .with_columns([
        pl.col("var").str.strip_chars("country_").cast(pl.Int64).alias("pos")
    ])
    .rename({"value": "country"})
)

# Join auteur ↔ pays par position
authors_countries = authors_long.join(
    countries_long, on=["year", "pos"], how="left"
)

# Pour chaque article, générer toutes les paires d'auteurs (combinations)
edges = (
    authors_countries
    .groupby("year", maintain_order=True)
    .agg([
        pl.struct(["author", "country"]).alias("nodes")
    ])
    .explode("nodes")
    .with_columns([
        pl.col("nodes").struct.field("author").alias("author"),
        pl.col("nodes").struct.field("country").alias("country")
    ])
)

# Maintenant, générons les combinaisons deux à deux par année
edges = (
    edges.groupby("year")
    .agg([
        pl.col("author").alias("authors"),
        pl.col("country").alias("countries")
    ])
    .select([
        "year",
        pl.struct(["authors", "countries"]).alias("pairs")
    ])
    .with_columns(
        pl.col("pairs").map_elements(
            lambda row: [
                {
                    "Source": row["authors"][i],
                    "Target": row["authors"][j],
                    "Type": "Undirected",
                    "Country_Source": row["countries"][i],
                    "Country_Target": row["countries"][j],
                    "Date": row["year"]
                }
                for i, j in itertools.combinations(range(len(row["authors"])), 2)
            ],
            return_dtype=pl.List(pl.Struct([
                pl.Field("Source", pl.Utf8),
                pl.Field("Target", pl.Utf8),
                pl.Field("Type", pl.Utf8),
                pl.Field("Country_Source", pl.Utf8),
                pl.Field("Country_Target", pl.Utf8),
                pl.Field("Date", pl.Int64),
            ]))
        ).alias("edges")
    )
    .explode("edges")
    .unnest("edges")
)


C:\Users\gabri\AppData\Local\Temp\ipykernel_21580\709791164.py:11: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  long_df = works.melt(


: 

In [ ]:
edges = edges[edges["Source"].notna()]
edges = edges[edges["Target"].notna()]
edges["count"] = 1
edges["now"] = 2025

In [6]:
edges.to_csv("data/works/edges/correction/edges.csv")

In [7]:
edges = edges[["Source", "Target"]]
edges.to_csv("data/works/edges/correction/edges_authors.csv", index = None)

## Statistical indicators

In [ ]:
g = nx.from_pandas_edgelist(edges, "Source", "Target")

average degree : 

In [14]:
print(dict(g.degree))

{'https://openalex.org/A5053584746': 8, 'https://openalex.org/A5087656450': 29, 'https://openalex.org/A5004753412': 29, 'https://openalex.org/A5019284502': 56, 'https://openalex.org/A5042962646': 4, 'https://openalex.org/A5109933717': 3, 'https://openalex.org/A5114172798': 3, 'https://openalex.org/A5066077197': 3, 'https://openalex.org/A5109981964': 3, 'https://openalex.org/A5053726981': 16, 'https://openalex.org/A5069136351': 43, 'https://openalex.org/A5090419729': 606, 'https://openalex.org/A5014692107': 323, 'https://openalex.org/A5105250606': 2, 'https://openalex.org/A5112151418': 23, 'https://openalex.org/A5056199248': 23, 'https://openalex.org/A5026696303': 35, 'https://openalex.org/A5039795707': 23, 'https://openalex.org/A5005357835': 111, 'https://openalex.org/A5062889773': 2, 'https://openalex.org/A5088686533': 2, 'https://openalex.org/A5060834868': 2, 'https://openalex.org/A5059319160': 136, 'https://openalex.org/A5110235872': 4, 'https://openalex.org/A5057824850': 12, 'https

In [10]:
sum(dict(g.degree).values())/g.number_of_nodes()

9.732848183080385

In [8]:
# top 10 centrality : 
centrality = nx.closeness_centrality(g)
top_10 = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:10]

KeyboardInterrupt: 

In [68]:
nx.density(g)

0.000153967703916368

In [71]:
largest_connected_component_g =  max(nx.connected_components(g), key = len)
nx.diameter(g.subgraph(largest_connected_component_g))

KeyboardInterrupt: 

In [72]:
nx.average_clustering(g)

0.8468244937581203